# Data Integrity and Volume Audit


This notebook performs a comprehensive audit of the data pipeline. It tracks the volume and integrity of the datasets across four key stages:

1. Broad Harvesting: Initial extraction of COVID-19 related news from GDELT.

2. Strict Filtering: Isolation of articles specifically addressing Mental Health using a refined keyword strategy.

3. Vectorization: Extraction of contextual embeddings (occurrences) from the filtered corpus.

4. Subspace Modeling: Final results derived from matrix decomposition.

The goal is to ensure consistency, identify any data loss, and validate the uniqueness of articles (URLs) across the temporal windows.

## 1. Imports and Configuration

In [3]:
import pandas as pd
import numpy as np
import os

# Define paths
PATHS = {
    "broad": r"../data/raw/harvest/_merged/spain_covid_broad_ALL.csv",
    "strict": r"../data/interim/datasets/spain_covidMHstrict_2020-03_2021-03_ALL.csv",
    "embeddings": r"../data/interim/embeddings/spain_covidMHstrict_occurrences_2020-03_2021-03_CLEANED.csv",
    "results": r"../data/phase3_results.parquet"
}

# Set display options
pd.options.display.float_format = '{:,.2f}'.format

## 2. Comprehensive Data Audit Script

In [4]:
def run_audit():
    print("🔍 STARTING DATA INTEGRITY AUDIT...\n")
    summary = []

    # --- PHASE 1: BROAD COVID CORPUS ---
    if os.path.exists(PATHS["broad"]):
        df_broad = pd.read_csv(PATHS["broad"], usecols=['url'])
        total_broad = len(df_broad)
        unique_broad = df_broad['url'].nunique()
        summary.append(["Phase 1: Broad COVID", total_broad, unique_broad])
        print(f"✅ Phase 1 Loaded: {total_broad:,} rows found.")
    else:
        print(f"❌ Broad file not found at: {PATHS['broad']}")

    # --- PHASE 1: STRICT MH FILTERED ---
    if os.path.exists(PATHS["strict"]):
        df_strict = pd.read_csv(PATHS["strict"], usecols=['url'])
        total_strict = len(df_strict)
        unique_strict = df_strict['url'].nunique()
        summary.append(["Phase 1: Strict MH", total_strict, unique_strict])
        print(f"✅ Phase 1 (Strict) Loaded: {total_strict:,} rows found.")
    else:
        print(f"❌ Strict file not found at: {PATHS['strict']}")

    # --- PHASE 2: EMBEDDINGS (OCCURRENCES) ---
    if os.path.exists(PATHS["embeddings"]):
        # Embeddings usually don't have URL, just count the rows (occurrences)
        df_emb = pd.read_csv(PATHS["embeddings"])
        total_emb = len(df_emb)
        summary.append(["Phase 2: Embeddings", total_emb, "N/A (Occurrences)"])
        print(f"✅ Phase 2 (Embeddings) Loaded: {total_emb:,} occurrences found.")
    else:
        print(f"❌ Embeddings file not found.")

    # --- PHASE 3: SUBSPACE RESULTS ---
    if os.path.exists(PATHS["results"]):
        df_res = pd.read_parquet(PATHS["results"])
        total_res_windows = len(df_res)
        summary.append(["Phase 3: Subspace Windows", total_res_windows, "N/A"])
        print(f"✅ Phase 3 (Results) Loaded: {total_res_windows} windows analyzed.")
    else:
        print(f"❌ Phase 3 results not found.")

    # Create Summary DataFrame
    audit_df = pd.DataFrame(summary, columns=["Phase", "Total Rows/Lines", "Unique Articles (URLs)"])
    
    print("\n" + "="*50)
    print("FINAL DATA FUNNEL SUMMARY")
    print("="*50)
    display(audit_df)
    
    # Calculate Ratios
    if len(summary) >= 2:
        retention = (summary[1][1] / summary[0][1]) * 100
        print(f"\n💡 Mental Health articles represent {retention:.2f}% of total COVID news.")
        
    if len(summary) >= 3:
        density = (summary[2][1] / summary[1][1])
        print(f"💡 On average, we extracted {density:.2f} occurrences (embeddings) per MH article.")

run_audit()

🔍 STARTING DATA INTEGRITY AUDIT...

✅ Phase 1 Loaded: 53,055 rows found.
✅ Phase 1 (Strict) Loaded: 2,156 rows found.
✅ Phase 2 (Embeddings) Loaded: 7,535 occurrences found.
✅ Phase 3 (Results) Loaded: 11 windows analyzed.

FINAL DATA FUNNEL SUMMARY


,Phase,Total Rows/Lines,Unique Articles (URLs)
0,Phase 1: Broad COVID,53055,53055
1,Phase 1: Strict MH,2156,2156
2,Phase 2: Embeddings,7535,N/A (Occurrences)
3,Phase 3: Subspace Windows,11,N/A



💡 Mental Health articles represent 4.06% of total COVID news.
💡 On average, we extracted 3.49 occurrences (embeddings) per MH article.


In [5]:
# Vamos a ver el tamaño real del archivo en disco (sin cargarlo en pandas)
import os
file_path = PATHS["strict"]
file_size = os.path.getsize(file_path) / (1024 * 1024) # Tamaño en MB
print(f"File size: {file_size:.2f} MB")

# Contar líneas reales del archivo (forma rápida de sistema operativo)
with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    line_count = sum(1 for line in f)
print(f"Real line count in file: {line_count}")

File size: 15.41 MB
Real line count in file: 71881


---

In [6]:
F_PATH = r"../data/interim/datasets/spain_covidMHstrict_2020-03_2021-03_ALL.csv"

In [8]:
# Carga el dataset de la Fase 1 (el filtrado de salud mental)
df_mh = pd.read_csv(F_PATH)

# 1. Comprobar si hay duplicados por URL (si tienes esa columna)
if 'url' in df_mh.columns:
    unique_articles = df_mh['url'].nunique()
    print(f"Total de líneas: {len(df_mh)}")
    print(f"Artículos únicos (por URL): {unique_articles}")
elif 'title' in df_mh.columns:
    unique_articles = df_mh['title'].nunique()
    print(f"Total de líneas: {len(df_mh)}")
    print(f"Artículos únicos (por Título): {unique_articles}")

# 2. Ver un ejemplo
print("\nPrimeras 15 líneas para ver la estructura:")
display(df_mh.head(15))

Total de líneas: 2156
Artículos únicos (por URL): 2156

Primeras 15 líneas para ver la estructura:


,title,newspaper,url,published_at,plain_text,keyword,relevance_score,source,mh_matches,covid_matches
0,Botón de pausa,eldiario.es,https://www.eldiario.es/canariasahora/canarias...,2020-03-04 23:15:00+00:00,Hace unos días The New York Times promocionaba...,"covid, coronavirus, pandemia",30.00,GDELT,salud mental,coronavirus
1,Coronavirus y niños : ¿ Qué contar y cómo ?,eldiario.es,https://www.eldiario.es/nidos/Coronavirus-nino...,2020-03-07 22:45:00+00:00,Coronavirus y niños: ¿Qué contar y cómo?\n\nEs...,"covid, coronavirus, pandemia",100.00,GDELT,psicologo;psicólogo,covid;covid-19;coronavirus
2,Voces femeninas : desde la China más profunda,lavanguardia.com,https://www.lavanguardia.com/cultura/culturas/...,2020-03-08 07:00:00+00:00,Ha querido la casualidad que en medio de la cr...,"covid, coronavirus, pandemia",40.00,GDELT,suicidio,coronavirus
3,Mujeres a pie de calle contras las violencias ...,lavanguardia.com,https://www.lavanguardia.com/vida/20200308/474...,2020-03-08 10:30:00+00:00,La doctora Lluïsa Garcia-Esteve lleva un lazo ...,"covid, coronavirus, pandemia",10.00,GDELT,salud mental;psiquiatra,coronavirus
4,Coronafobia,20minutos.es,https://www.20minutos.es/opinion/inaki-ortega-...,2020-03-09 06:30:00+00:00,Me cuentan mis hijos que una serpiente se dio ...,"covid, coronavirus, pandemia",30.00,GDELT,psiquiatria;psiquiatría,covid;covid-19;coronavirus;pandemia
5,Sobre el coronavirus ... Yo solo me fío de Fer...,20minutos.es,https://www.20minutos.es/opinion/juan-carlos-b...,2020-03-10 06:00:00+00:00,El miedo es libre. Y también es muy contagioso...,"covid, coronavirus, pandemia",70.00,GDELT,ansiedad,coronavirus;pandemia
6,Europa se prepara para varios meses de medidas...,eldiario.es,https://www.eldiario.es/theguardian/Europa-pre...,2020-03-10 08:45:00+00:00,Europa se prepara para varios meses de medidas...,"covid, coronavirus, pandemia",100.00,GDELT,ansiedad,coronavirus;pandemia;cuarentena
7,Crimen Guardia Urbana : La acusación arrolla a...,lavanguardia.com,https://www.lavanguardia.com/sucesos/20200310/...,2020-03-10 22:00:00+00:00,Rosa Peral contrató a un psicólogo forense par...,"covid, coronavirus, pandemia",20.00,GDELT,ansiedad;psicologo;psicólogo,coronavirus
8,Bélgica desaconseja actos de más de 1 . 000 pe...,europapress.es,https://www.europapress.es/internacional/notic...,2020-03-10 16:15:00+00:00,"El coronavirus en España y en el mundo, en dir...","covid, coronavirus, pandemia",60.00,GDELT,ansiedad,covid;covid-19;coronavirus
9,¿ Regreso a 2008 ? El pánico en la deuda amena...,elconfidencial.com,https://www.elconfidencial.com/economia/2020-0...,2020-03-10 06:30:00+00:00,El hundimiento de las bolsas ha desencadenado ...,"covid, coronavirus, pandemia",30.00,GDELT,estres;estrés,coronavirus
